# Finding sequences a substitution or two apart

`portmanteau.sequence.hamming_pairs` finds pairs of equal-length sequences within a small Hamming
distance — **within one dataframe**, or **across two**.

In [1]:
import pandas as pd

from portmanteau.sequence.hamming_pairs import (
    find_hamming1_pairs,
    find_hamming1_pairs_between,
    find_hamming_pairs,
    find_hamming_pairs_between,
)

## Two small datasets

Short enough to check by eye. Things to notice as we go:

- `CASSLF`/`CASSPF` are one substitution apart **inside** `pdf_a`, and `CASSRF`/`CASSTF` are a pair
  **inside** `pdf_b`.
- `CASSPF` appears in **both** frames.
- `CAVVLF` is **two** substitutions from `CASSLF`.
- `CAWWGHK` is seven characters, and sequences of different lengths are never paired.
- donor `d3` appears only in `pdf_b`.

In [2]:
pdf_a = pd.DataFrame({
    "cdr3":  ["CASSLF", "CASSPF", "CASSYF", "CAWWGHK"],
    "donor": ["d1",     "d1",     "d2",     "d2"],
})
pdf_b = pd.DataFrame({
    "cdr3":  ["CASSRF", "CASSTF", "CASSPF", "CAVVLF"],
    "donor": ["d1",     "d2",     "d3",     "d1"],
})
display(pdf_a, pdf_b)

,cdr3,donor
0,CASSLF,d1
1,CASSPF,d1
2,CASSYF,d2
3,CAWWGHK,d2


,cdr3,donor
0,CASSRF,d1
1,CASSTF,d2
2,CASSPF,d3
3,CAVVLF,d1


## Within one frame

Every pair differing at exactly one position, each pair reported once. `row_i` and `row_j` are
index *labels*, not positions, so they still point at the right rows if the frame has an index of
its own. `CAWWGHK` pairs with nothing: wrong length.

In [3]:
find_hamming1_pairs(pdf_a, "cdr3")

,row_i,row_j,cdr3_i,cdr3_j,seq_length,differing_position
0,0,1,CASSLF,CASSPF,6,4
1,0,2,CASSLF,CASSYF,6,4
2,1,2,CASSPF,CASSYF,6,4


## Across two frames

One sequence from each. Note what is **absent**: `CASSLF`/`CASSPF` is a pair inside `pdf_a` and
`CASSRF`/`CASSTF` is one inside `pdf_b`, so neither appears here. `row_i` is always from `pdf_a`,
`row_j` always from `pdf_b`.

In [4]:
find_hamming1_pairs_between(pdf_a, pdf_b, "cdr3")

,row_i,row_j,cdr3_i,cdr3_j,seq_length,differing_position
0,0,0,CASSLF,CASSRF,6,4
1,0,1,CASSLF,CASSTF,6,4
2,0,2,CASSLF,CASSPF,6,4
3,1,0,CASSPF,CASSRF,6,4
4,1,1,CASSPF,CASSTF,6,4
5,2,0,CASSYF,CASSRF,6,4
6,2,1,CASSYF,CASSTF,6,4
7,2,2,CASSYF,CASSPF,6,4


### The same sequence in both frames

`CASSPF` is in `pdf_a` and in `pdf_b`. That is a distance of **0**, not 1, so it is missing from
the result above. It is still worth knowing about — unlike two identical rows inside one frame,
which are just the same sequence twice, the same sequence in two sources is a real observation.

`exact=False` returns everything *up to* `distance` and adds a `hamming_distance` column:

In [5]:
pdf_pairs = find_hamming_pairs_between(pdf_a, pdf_b, "cdr3", distance=1, exact=False)
pdf_pairs[pdf_pairs.hamming_distance == 0]

,row_i,row_j,cdr3_i,cdr3_j,seq_length,differing_positions,hamming_distance
0,1,2,CASSPF,CASSPF,6,(),0


## Requiring the pair to share something

`group_cols` names columns that must match. Across two frames, a group present in only one of them
pairs with nothing — donor `d3` is only in `pdf_b`, so `CASSPF` drops out entirely here.

In [6]:
find_hamming1_pairs_between(pdf_a, pdf_b, "cdr3", group_cols=["donor"])

,row_i,row_j,cdr3_i,cdr3_j,seq_length,differing_position
0,0,0,CASSLF,CASSRF,6,4
1,1,0,CASSPF,CASSRF,6,4
2,2,1,CASSYF,CASSTF,6,4


Grouping is also the main lever against the cost: searching within each of *k* groups is
`k * (n/k)^2` work rather than `n^2`. If the sequences carry a natural partition — one subject, one
sample, one repertoire — grouping on it is both cheaper and usually the question you meant.

## Carrying other columns onto the pairs

`annotation_columns` adds `{col}_i` and `{col}_j`, each taken from the frame that side came from.

In [7]:
find_hamming1_pairs_between(pdf_a, pdf_b, "cdr3", annotation_columns=["donor"])

,row_i,row_j,cdr3_i,cdr3_j,seq_length,differing_position,donor_i,donor_j
0,0,0,CASSLF,CASSRF,6,4,d1,d1
1,0,1,CASSLF,CASSTF,6,4,d1,d2
2,0,2,CASSLF,CASSPF,6,4,d1,d3
3,1,0,CASSPF,CASSRF,6,4,d1,d1
4,1,1,CASSPF,CASSTF,6,4,d1,d2
5,2,0,CASSYF,CASSRF,6,4,d2,d1
6,2,1,CASSYF,CASSTF,6,4,d2,d2
7,2,2,CASSYF,CASSPF,6,4,d2,d3


## Further than one substitution

`find_hamming_pairs` and `find_hamming_pairs_between` take a `distance`. With `exact=True` (the
default) you get pairs at exactly that distance — so `CASSLF`/`CAVVLF` appears and the many
distance-1 pairs do not. The differing positions come back as a tuple, `differing_positions`,
rather than the scalar `differing_position` the distance-1 wrappers report.

In [8]:
find_hamming_pairs_between(pdf_a, pdf_b, "cdr3", distance=2)

,row_i,row_j,cdr3_i,cdr3_j,seq_length,differing_positions
0,0,3,CASSLF,CAVVLF,6,"(2, 3)"


## One thing to watch

**Pair counts grow quadratically.** Ten times the sequences is a hundred times the pairs, which is
easy to underestimate. `max_pairs` turns that into an error rather than a dead machine, and
`iter_hamming_pairs` / `iter_hamming_pairs_between` stream the result in chunks when it really is
that large.

In [9]:
try:
    find_hamming1_pairs_between(pdf_a, pdf_b, "cdr3", max_pairs=2)
except ValueError as error:
    print(error)

more than max_pairs=2 pairs; pair counts grow quadratically with input size. Narrow the search with group_cols, or stream the result with iter_hamming_pairs.


## Summary

| | within one frame | across two |
|---|---|---|
| exactly one substitution | `find_hamming1_pairs` | `find_hamming1_pairs_between` |
| any distance | `find_hamming_pairs` | `find_hamming_pairs_between` |
| streamed in chunks | `iter_hamming_pairs` | `iter_hamming_pairs_between` |

There is one implementation underneath: the one-frame search is the two-frame routine with both
sides the same list, which costs nothing extra.